<a href="https://colab.research.google.com/github/shrivastavay882-glitch/Healthcare_Analysis/blob/main/Healthcare_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [18]:
import pandas as pd
import numpy as np


df = pd.read_csv('/content/synthetic_hospital_patient_admissions.csv')
df.info()    # Medical_speciality shows null values present
df.head()

# Handling null values in Medical_speciality
df['Medical_Specialty'] = df['Medical_Specialty'].fillna('Unspecified')
df.info()

#  Standardize Text Casing and Remove Invisible White Spaces
df['Gender'] = df['Gender'].str.strip().str.title()
df['Admission_Type'] = df['Admission_Type'].str.strip().str.title()
df['Primary_Diagnosis'] = df['Primary_Diagnosis'].str.strip().str.title()
df['Medical_Specialty'] = df['Medical_Specialty'].str.strip().str.title()
df['Discharge_Disposition'] = df['Discharge_Disposition'].str.strip().str.title()
df['Readmitted_30_Days'] = df['Readmitted_30_Days'].str.strip().str.title()


#  Lifespan Clinical Age Groups (0 to 95+)
age_bins = [-1, 1, 12, 19, 39, 59, 79, np.inf]   # starting from -1 so we can include infants of age 0
age_labels = ['0-1 (Infant)', '2-12 (Pediatric)', '13-19 (Teenager)',
              '20-39 (Young Adult)', '40-59 (Middle-Aged)',
              '60-79 (Senior Citizen)', '80+ (Advanced Geriatric)']
df['Age_Group'] = pd.cut(df['Age'], bins=age_bins, labels=age_labels)

# converting the cleaned data into csv
df.to_csv('Cleaned_hospital_admission_csv',index=False)



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4000 entries, 0 to 3999
Data columns (total 12 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Patient_ID              4000 non-null   object 
 1   Age                     4000 non-null   int64  
 2   Gender                  4000 non-null   object 
 3   Admission_Type          4000 non-null   object 
 4   Primary_Diagnosis       4000 non-null   object 
 5   Medical_Specialty       3920 non-null   object 
 6   Time_in_Hospital        4000 non-null   int64  
 7   Lab_Procedures_Count    4000 non-null   int64  
 8   Medications_Prescribed  4000 non-null   int64  
 9   Total_Charges           4000 non-null   float64
 10  Discharge_Disposition   4000 non-null   object 
 11  Readmitted_30_Days      4000 non-null   object 
dtypes: float64(1), int64(4), object(7)
memory usage: 375.1+ KB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4000 entries, 0 to 3999
Data columns 

In [23]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('Healthcare_Analysis')
df = pd.read_csv('/content/Cleaned_hospital_admission_csv')
df.to_sql('hospital_admissions', conn, if_exists='replace', index=False)


analytics_query = """
WITH Specialty_Metrics AS (
    SELECT
        Medical_Specialty,
        COUNT(Patient_ID) AS total_patients,
        SUM(CASE WHEN Readmitted_30_Days = 'Yes' THEN 1 ELSE 0 END) AS total_readmissions,
        SUM(Total_Charges) AS gross_revenue_inr,
        AVG(Time_in_Hospital) AS avg_days_stay
    FROM hospital_admissions
    GROUP BY Medical_Specialty
)
SELECT
    Medical_Specialty,
    total_patients,
    total_readmissions,
    ROUND((total_readmissions * 100.0 / total_patients), 2) AS readmission_rate_pct,
    ROUND(gross_revenue_inr/ 10000000.0, 2) AS total_revenue_inr_cr,
    ROUND(avg_days_stay, 1) AS average_stay_days
FROM Specialty_Metrics
ORDER BY readmission_rate_pct DESC;
"""

pd.read_sql_query(analytics_query, conn)

,Medical_Specialty,total_patients,total_readmissions,readmission_rate_pct,total_revenue_inr_cr,average_stay_days
0,Cardiology,653,87,13.32,38.54,4.7
1,Orthopedics,600,72,12.00,29.67,5.2
2,General Medicine,2062,240,11.64,110.03,5.5
3,Unspecified,80,9,11.25,3.86,4.5
4,Endocrinology,464,46,9.91,16.36,4.9
5,Pediatrics,141,10,7.09,5.74,4.6
